In [ ]:
# CELL 1 — IMPORTS
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import shutil

In [ ]:
# CELL 2 — PATHS
DATASET_PATH = "/kaggle/input/datasets/mithilesh2303/deepfashion/In-shop Clothes Retrieval Benchmark"

EVAL_FILE  = os.path.join(DATASET_PATH, "Eval", "list_eval_partition.txt")
BBOX_FILE  = os.path.join(DATASET_PATH, "Anno", "list_bbox_inshop.txt")
OUTPUT_DIR = "/kaggle/working/cropped_images"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Eval file exists:", os.path.exists(EVAL_FILE))
print("BBox file exists:", os.path.exists(BBOX_FILE))

In [ ]:
# CELL 3 — PARSE list_eval_partition.txt
# Format (after 2 header lines):
#   image/relative/path   id_XXXXXX   train|query|gallery

data = []
with open(EVAL_FILE, "r") as f:
    lines = f.readlines()

print("Header line 0:", lines[0].strip())
print("Header line 1:", lines[1].strip())
print("First data line:", lines[2].strip())

for line in lines[2:]:
    parts = line.strip().split()
    if len(parts) < 3:
        continue
    image_relative_path = parts[0]
    item_id             = parts[1]
    split               = parts[2]
    full_image_path     = os.path.join(DATASET_PATH, "Img", "img", image_relative_path)
    save_name           = image_relative_path.replace("/", "_")
    data.append({
        "image_path"    : full_image_path,
        "relative_path" : image_relative_path,
        "save_name"     : save_name,
        "item_id"       : item_id,
        "split"         : split
    })

df = pd.DataFrame(data)
print(df.head())
print(f"\nTotal images: {len(df)}")

In [ ]:
# CELL 4 — VERIFY SPLIT COUNTS
split_counts = df["split"].value_counts()
print(split_counts)

expected = {"train": 25882, "query": 14218, "gallery": 12612}
for s, exp in expected.items():
    got    = split_counts.get(s, 0)
    diff   = got - exp
    status = "OK" if abs(diff) < 200 else "CHECK"
    print(f"  {s:10s}  got={got:6d}  expected≈{exp:6d}  diff={diff:+d}  [{status}]")

unexpected = set(df["split"].unique()) - {"train", "query", "gallery"}
if unexpected:
    print(f"WARNING: unexpected split labels found: {unexpected}")
else:
    print("\nAll split labels are valid (train / query / gallery).")

In [ ]:
# CELL 5 — PARSE list_bbox_inshop.txt
# Format (after 2 header lines):
#   image/relative/path   x_1  y_1  x_2  y_2
# Coordinates are absolute pixel values.

bbox_data = []
with open(BBOX_FILE, "r") as f:
    bbox_lines = f.readlines()

print("BBox header line 0:", bbox_lines[0].strip())
print("BBox header line 1:", bbox_lines[1].strip())
print("First data line:   ", bbox_lines[2].strip())

for line in bbox_lines[2:]:
    parts = line.strip().split()
    if len(parts) < 5:
        continue
    bbox_data.append({
        "relative_path" : parts[0],
        "gt_x1"         : int(parts[1]),
        "gt_y1"         : int(parts[2]),
        "gt_x2"         : int(parts[3]),
        "gt_y2"         : int(parts[4])
    })

bbox_df = pd.DataFrame(bbox_data)
print(f"\nTotal GT bbox entries: {len(bbox_df)}")
print(bbox_df.head())

In [ ]:
# CELL 6 — MERGE GT BOXES INTO MAIN DF
df = df.merge(bbox_df, on="relative_path", how="left")

missing_gt = df["gt_x1"].isna().sum()
print(f"Rows with GT box   : {len(df) - missing_gt}")
print(f"Rows WITHOUT GT box: {missing_gt}  ← these will fallback to full image")
print()
print(df[["relative_path", "gt_x1", "gt_y1", "gt_x2", "gt_y2"]].head())

In [ ]:
# CELL 7 — VALIDATE GT BOX DIMENSIONS
# Sanity check: boxes should be non-degenerate (x2 > x1, y2 > y1, min size > 10px)

has_gt = df["gt_x1"].notna()
sub    = df[has_gt].copy()

sub["w"] = sub["gt_x2"] - sub["gt_x1"]
sub["h"] = sub["gt_y2"] - sub["gt_y1"]

bad_boxes = sub[(sub["w"] <= 10) | (sub["h"] <= 10)]
print(f"GT boxes with w or h <= 10px (degenerate): {len(bad_boxes)}")
if len(bad_boxes) > 0:
    print(bad_boxes[["relative_path", "gt_x1", "gt_y1", "gt_x2", "gt_y2", "w", "h"]].head())

print(f"\nBox width  — min: {sub['w'].min():.0f}  median: {sub['w'].median():.0f}  max: {sub['w'].max():.0f}")
print(f"Box height — min: {sub['h'].min():.0f}  median: {sub['h'].median():.0f}  max: {sub['h'].max():.0f}")

In [ ]:
# CELL 8 — VISUALISE GT CROP ON A SAMPLE IMAGE
from PIL import ImageDraw

sample = df[df["gt_x1"].notna()].iloc[0]
image  = Image.open(sample["image_path"]).convert("RGB")
x1, y1, x2, y2 = int(sample["gt_x1"]), int(sample["gt_y1"]), int(sample["gt_x2"]), int(sample["gt_y2"])

# Clamp to image bounds
W, H   = image.size
x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
cropped = image.crop((x1, y1, x2, y2))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(cropped)
axes[1].set_title(f"GT Crop  [{x1},{y1},{x2},{y2}]")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# CELL 9 — CLEAN OUTPUT DIRECTORY
shutil.rmtree("/kaggle/working/cropped_images", ignore_errors=True)
os.makedirs("/kaggle/working/cropped_images", exist_ok=True)
print("Output directory cleaned and ready.")

In [ ]:
# CELL 10 — MAIN CROP LOOP (all splits: train + query + gallery)
# For each image:
#   1. Use GT bbox if available — clamp to image bounds
#   2. If GT bbox missing or degenerate → fallback to full image (tracked)
#   3. Save to OUTPUT_DIR using save_name

MIN_BOX_DIM = 10   # minimum width/height in pixels to treat a GT box as valid

failed        = []
fallback_used = []
batch_size    = 500
total_images  = len(df)

print(f"Processing {total_images} images in batches of {batch_size}...")

for start in range(0, total_images, batch_size):
    end      = min(start + batch_size, total_images)
    batch_df = df.iloc[start:end]
    print(f"\nBatch {start}–{end}")

    for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), leave=False):
        img_path  = row["image_path"]
        save_name = row["save_name"]
        save_path = os.path.join(OUTPUT_DIR, save_name)

        try:
            image = Image.open(img_path).convert("RGB")
            W, H  = image.size

            use_gt = (
                pd.notna(row["gt_x1"]) and
                (row["gt_x2"] - row["gt_x1"]) > MIN_BOX_DIM and
                (row["gt_y2"] - row["gt_y1"]) > MIN_BOX_DIM
            )

            if use_gt:
                x1 = max(0, int(row["gt_x1"]))
                y1 = max(0, int(row["gt_y1"]))
                x2 = min(W, int(row["gt_x2"]))
                y2 = min(H, int(row["gt_y2"]))
                cropped = image.crop((x1, y1, x2, y2))
            else:
                cropped = image
                fallback_used.append(img_path)

            cropped.save(save_path)

        except Exception as e:
            failed.append((img_path, str(e)))

    print(f"  Done. Fallbacks so far: {len(fallback_used)}  |  Failures so far: {len(failed)}")

print("\n=== CROP LOOP COMPLETE ===")
print(f"Total images processed : {total_images}")
print(f"GT box used            : {total_images - len(fallback_used) - len(failed)}")
print(f"Fallback (full image)  : {len(fallback_used)}")
print(f"Failed (errors)        : {len(failed)}")
print(f"Files in output dir    : {len(os.listdir(OUTPUT_DIR))}")

In [ ]:
# CELL 11 — INSPECT FAILURES (if any)
print(f"Failed images ({len(failed)}):")
for path, err in failed[:10]:
    print(f"  {path}  →  {err}")

In [ ]:
# CELL 12 — BUILD cropped_metadata.csv
cropped_data    = []
missing_on_disk = 0

for idx, row in df.iterrows():
    save_name    = row["save_name"]
    cropped_path = os.path.join(OUTPUT_DIR, save_name)
    on_disk      = os.path.exists(cropped_path)
    if not on_disk:
        missing_on_disk += 1

    cropped_data.append({
        "cropped_image_path" : cropped_path,
        "relative_path"      : row["relative_path"],
        "item_id"            : row["item_id"],
        "split"              : row["split"],
        "fallback_used"      : row["image_path"] in fallback_used,
        "on_disk"            : on_disk,
        "gt_x1"              : row["gt_x1"],
        "gt_y1"              : row["gt_y1"],
        "gt_x2"              : row["gt_x2"],
        "gt_y2"              : row["gt_y2"]
    })

cropped_df = pd.DataFrame(cropped_data)

print(cropped_df.head())
print(f"\nTotal rows      : {len(cropped_df)}")
print(f"On disk         : {cropped_df['on_disk'].sum()}")
print(f"Missing on disk : {missing_on_disk}")
print()
print("Per-split breakdown:")
print(cropped_df.groupby("split")[["on_disk", "fallback_used"]].sum())

cropped_df.to_csv("/kaggle/working/cropped_metadata.csv", index=False)
print("\nSaved: /kaggle/working/cropped_metadata.csv")

In [ ]:
# CELL 13 — BUILD all_metadata.csv
all_meta_data = []

for idx, row in df.iterrows():
    save_name    = row["save_name"]
    cropped_path = os.path.join(OUTPUT_DIR, save_name)
    all_meta_data.append({
        "image_path"         : row["image_path"],
        "relative_path"      : row["relative_path"],
        "item_id"            : row["item_id"],
        "split"              : row["split"],
        "cropped_image_path" : cropped_path,
        "fallback_used"      : row["image_path"] in fallback_used,
        "gt_x1"              : row["gt_x1"],
        "gt_y1"              : row["gt_y1"],
        "gt_x2"              : row["gt_x2"],
        "gt_y2"              : row["gt_y2"]
    })

all_df = pd.DataFrame(all_meta_data)

print("=== all_metadata.csv summary ===")
print(f"Total rows             : {len(all_df)}")
print(f"Unique item_ids        : {all_df['item_id'].nunique()}")
print()
print("Split counts:")
print(all_df["split"].value_counts())

all_df.to_csv("/kaggle/working/all_metadata.csv", index=False)
print("\nSaved: /kaggle/working/all_metadata.csv")

In [ ]:
# CELL 14 — VISUALISE SAMPLE CROPS (one per split)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, split in enumerate(["train", "query", "gallery"]):
    row     = cropped_df[cropped_df["split"] == split].iloc[0]
    img_pth = row["cropped_image_path"]
    if os.path.exists(img_pth):
        img = Image.open(img_pth)
        axes[i].imshow(img)
        axes[i].set_title(f"{split}\n{row['item_id']}")
    else:
        axes[i].set_title(f"{split} — file missing")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# CELL 15 — PACKAGE INTO final_dataset/
FINAL_DIR = "/kaggle/working/final_dataset"

shutil.rmtree(FINAL_DIR, ignore_errors=True)
for old_zip in ["/kaggle/working/final_dataset.zip", "/kaggle/working/processed_dataset.zip"]:
    if os.path.exists(old_zip):
        os.remove(old_zip)

os.makedirs(FINAL_DIR, exist_ok=True)

shutil.copy("/kaggle/working/cropped_metadata.csv", FINAL_DIR)
shutil.copy("/kaggle/working/all_metadata.csv",     FINAL_DIR)
shutil.copytree(
    "/kaggle/working/cropped_images",
    os.path.join(FINAL_DIR, "cropped_images"),
    dirs_exist_ok=True
)

print("final_dataset/ contents:")
for f in sorted(os.listdir(FINAL_DIR)):
    print(" ", f)
print(f"  cropped_images/  ({len(os.listdir(os.path.join(FINAL_DIR, 'cropped_images')))} files)")

In [ ]:
# CELL 16 — CREATE ZIP
shutil.make_archive("/kaggle/working/final_dataset", "zip", FINAL_DIR)
size_mb = os.path.getsize("/kaggle/working/final_dataset.zip") / (1024**2)
print(f"ZIP created: /kaggle/working/final_dataset.zip  ({size_mb:.1f} MB)")

In [ ]:
# CELL 17 — FINAL SANITY CHECK
print("=== FINAL SANITY CHECK ===")
print()

n_cropped = len(os.listdir(OUTPUT_DIR))
n_df      = len(df)
print(f"Images in df              : {n_df}")
print(f"Files saved in output dir : {n_cropped}")
print(f"Match                     : {n_cropped == n_df - len(failed)}")
print()

cm_df   = pd.read_csv("/kaggle/working/cropped_metadata.csv")
all_csv = pd.read_csv("/kaggle/working/all_metadata.csv")
print(f"cropped_metadata.csv rows : {len(cm_df)}")
print(f"all_metadata.csv rows     : {len(all_csv)}")
print()

print("Splits in all_metadata.csv:")
print(all_csv["split"].value_counts())
print()

print(f"Unique item_ids in all_metadata.csv : {all_csv['item_id'].nunique()}")
print(f"Unique item_ids in source df        : {df['item_id'].nunique()}")
ids_match = set(all_csv["item_id"]) == set(df["item_id"])
print(f"All item_ids present in CSV         : {ids_match}")
print()

print("Path consistency spot-check (5 rows):")
for _, r in all_csv.sample(5, random_state=0).iterrows():
    expected_name = r["relative_path"].replace("/", "_")
    actual_name   = os.path.basename(r["cropped_image_path"])
    ok = expected_name == actual_name
    print(f"  {'OK' if ok else 'MISMATCH'}  {actual_name}")

print()
print(f"GT box coverage : {cm_df['gt_x1'].notna().sum()} / {len(cm_df)}")
print(f"Fallback used   : {cm_df['fallback_used'].sum()}")